# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² Mental Health Survey Dataset using the `mlcroissant` library and references all RecordSets, Fields, and Columns explicitly by their `@id`s.

### Dataset Source
The dataset is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print Brief Description
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available RecordSets, their fields, and the relevant `@id` values.

**Important:** All references to schema elements below use their `@id` as required.

In [ ]:
# List all available RecordSets in the dataset, displaying their @id and fields

print("Available RecordSets and their fields:\n")
recordsets = list(dataset.record_sets)
for rs in recordsets:
    print(f"- RecordSet: {rs.id}")
    print(f"    Name: {rs.name}")
    print(f"    Description: {rs.description}")
    print(f"    Fields:")
    for f in rs.fields:
        print(f"      - {f.id} (name: {f.name}, dataType: {f.data_type})")
    print("")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Below we:
- Gather all `@id`s for available record sets
- Iterate through RecordSet IDs, loading their records as DataFrames for easy manipulation
- Display available columns for inspection

**Note:** All uses of schema entities reference their `@id` fields only.

In [ ]:
# Extract data for all RecordSets by their @id

# Get all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dfs = {}

for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(recs)

# Display columns and a preview for the first recordset
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in RecordSet {main_rs_id}:")
    print(dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())
else:
    print("No RecordSets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing and EDA steps. In accordance with schema structure, all field accesses use their `@id`.

For demonstration, let's select a numeric field (such as the PHQ-9 score) by its field `@id`, filter/normalize it, and group by a demographic field.

In [ ]:
# Example: Analyzing the PHQ-9 score
# Update field IDs here if RecordSet schema changes

# Set below based on printout of available columns
phq9_field_id = 'phq_9_total_score' # Example: adjust to real @id if needed
group_field_id = 'gender'          # Example: adjust to real @id if needed

main_df = dfs[main_rs_id]

# Make sure selected field exists
if phq9_field_id in main_df.columns:
    # Remove outliers above the 99th percentile (if any)
    threshold = main_df[phq9_field_id].quantile(0.99)
    filtered_df = main_df[main_df[phq9_field_id] <= threshold].copy()
    print(f"Filtered to <=99th percentile: threshold={threshold:.1f}")

    # Normalize PHQ-9
    mean = filtered_df[phq9_field_id].mean()
    std = filtered_df[phq9_field_id].std()
    filtered_df[phq9_field_id + '_normalized'] = (filtered_df[phq9_field_id] - mean) / std
    print("\nSample normalized data:")
    display(filtered_df[[phq9_field_id, phq9_field_id + '_normalized']].head())

    # Group and aggregate by gender (or another group)
    if group_field_id in filtered_df:
        grouped = filtered_df.groupby(group_field_id)[phq9_field_id].mean().reset_index()
        print(f"\nMean PHQ-9 score by group ({group_field_id}):")
        display(grouped)
else:
    print(f"Field '{phq9_field_id}' not found in columns: {main_df.columns.tolist()}")

## 5. Visualization

Visualize the distribution of the PHQ-9 total score and its breakdown by group (for example, by gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if phq9_field_id in main_df:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[phq9_field_id].dropna(), bins=15, kde=True, color='purple')
    plt.title('Distribution of PHQ-9 Total Score')
    plt.xlabel('PHQ-9 Total Score (@id: phq_9_total_score)')
    plt.ylabel('Number of Participants')
    plt.show()

    if group_field_id in main_df:
        plt.figure(figsize=(7,4))
        sns.boxplot(data=main_df, x=group_field_id, y=phq9_field_id)
        plt.title(f'PHQ-9 Score by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel('PHQ-9 Total Score')
        plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² Mental Health Survey Dataset using its Croissant schema via `mlcroissant`, explored its structure, extracted records for each RecordSet by `@id`, and performed basic exploratory analysis and visualization. All schema elements were referenced by their `@id` fields for clarity and best practices.

This approach ensures robust, reproducible analysis and enables working across evolving schema versions. Consider further analyzing groups, sub-populations, or conducting deeper statistical inference using the field and RecordSet `@id`s as demonstrated.